In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import joblib

In [ ]:
df = pd.read_excel("Telco_customer_churn.xlsx")

In [ ]:
drop_cols = [
    "CustomerID", "Count", "Country", "State", "City",
    "Zip Code", "Lat Long", "Latitude", "Longitude",
    "Churn Score", "CLTV", "Churn Reason", "Churn Value", "Gender"
]
df.drop(columns=drop_cols, inplace=True)

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
df["Senior Citizen"].unique()
df["Senior Citizen"] = df["Senior Citizen"].replace("No", 0)
df["Senior Citizen"] = df["Senior Citizen"].replace("Yes", 1)

In [ ]:
df["Partner"].unique()
df["Partner"] = df["Partner"].replace("Yes", 1)
df["Partner"] = df["Partner"].replace("No", 0)

In [ ]:
df["Dependents"].unique()
df["Dependents"] = df["Dependents"].replace("Yes", 1)
df["Dependents"] = df["Dependents"].replace("No", 0)

In [ ]:
df["Phone Service"].unique()
df["Phone Service"] = df["Phone Service"].replace("Yes", 1)
df["Phone Service"] = df["Phone Service"].replace("No", 0)

In [ ]:
df["Multiple Lines"].unique()
df["Multiple Lines"] = df["Multiple Lines"].replace("Yes", 1)
df["Multiple Lines"] = df["Multiple Lines"].replace("No", 0)
df["Multiple Lines"] = df["Multiple Lines"].replace("No phone service", 2)

In [ ]:
df["Internet Service"].unique()
df["Internet Service"] = df["Internet Service"].replace("DSL", 1)
df["Internet Service"] = df["Internet Service"].replace("No", 0)
df["Internet Service"] = df["Internet Service"].replace("Fiber optic", 2)

In [ ]:
df["Online Security"].unique()
df["Online Security"] = df["Online Security"].replace("Yes", 1)
df["Online Security"] = df["Online Security"].replace("No", 0)
df["Online Security"] = df["Online Security"].replace("No internet service", 2)

In [ ]:
df["Online Backup"].unique()
df["Online Backup"] = df["Online Backup"].replace("Yes", 1)
df["Online Backup"] = df["Online Backup"].replace("No", 0)
df["Online Backup"] = df["Online Backup"].replace("No internet service", 2)

In [ ]:
df["Device Protection"].unique()
df["Device Protection"] = df["Device Protection"].replace("Yes", 1)
df["Device Protection"] = df["Device Protection"].replace("No", 0)
df["Device Protection"] = df["Device Protection"].replace("No internet service", 2)

In [ ]:
df["Tech Support"].unique()
df["Tech Support"] = df["Tech Support"].replace("Yes", 1)
df["Tech Support"] = df["Tech Support"].replace("No", 0)
df["Tech Support"] = df["Tech Support"].replace("No internet service", 2)

In [ ]:
df["Streaming TV"].unique()
df["Streaming TV"] = df["Streaming TV"].replace("Yes", 1)
df["Streaming TV"] = df["Streaming TV"].replace("No", 0)
df["Streaming TV"] = df["Streaming TV"].replace("No internet service", 2)

In [ ]:
df["Streaming Movies"].unique()
df["Streaming Movies"] = df["Streaming Movies"].replace("Yes", 1)
df["Streaming Movies"] = df["Streaming Movies"].replace("No", 0)
df["Streaming Movies"] = df["Streaming Movies"].replace("No internet service", 2)

In [ ]:
df['Contract'].unique()
df['Contract'] = df['Contract'].replace('Month-to-month', 1)
df['Contract'] = df['Contract'].replace('One year', 2)
df['Contract'] = df['Contract'].replace('Two year', 3)

In [ ]:
df['Paperless Billing'].unique()
df['Paperless Billing'] = df['Paperless Billing'].replace('Yes', 1)
df['Paperless Billing'] = df['Paperless Billing'].replace('No', 0)

In [ ]:
df["Payment Method"].unique()
df["Payment Method"] = df["Payment Method"].replace("Electronic check", 1)
df["Payment Method"] = df["Payment Method"].replace("Mailed check", 2)
df["Payment Method"] = df["Payment Method"].replace("Bank transfer (automatic)", 3)
df["Payment Method"] = df["Payment Method"].replace("Credit card (automatic)", 4)

In [ ]:
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
df["Total Charges"].fillna(0, inplace=True)

scaler = MinMaxScaler()
df[["Tenure Months Scaled", "Total Charges Scaled"]] = scaler.fit_transform(
    df[["Tenure Months", "Total Charges"]]
)

df.drop(columns=["Tenure Months", "Total Charges"], inplace=True)


In [ ]:
feature_cols = [
    'Senior Citizen', 'Partner', 'Dependents', 'Phone Service',
    'Multiple Lines', 'Internet Service', 'Online Security',
    'Online Backup', 'Device Protection', 'Tech Support',
    'Streaming TV', 'Streaming Movies', 'Contract',
    'Paperless Billing', 'Payment Method',
    'Monthly Charges', 'Tenure Months Scaled', 'Total Charges Scaled'
]

X = df[feature_cols]
df["Churn"] = df["Churn Label"].map({"Yes": 1, "No": 0})

y = df["Churn"]


In [ ]:
print(df.isna().sum())

Senior Citizen          0
Partner                 0
Dependents              0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Churn Label             0
Tenure Months Scaled    0
Total Charges Scaled    0
Churn                   0
dtype: int64


In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(X)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X = imputer.fit_transform(X)
X = scaler.fit_transform(X)


In [ ]:
import tensorflow as tf
from tensorflow import keras

# Define a simple sequential model (example architecture)
model = keras.Sequential([
    keras.layers.Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),

    keras.layers.Dense(64, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),

    keras.layers.Dense(1, activation="sigmoid")
])


model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss="binary_crossentropy",
    metrics=[keras.metrics.AUC(name="auc")]
)

from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=110,
    batch_size=32,
    callbacks=[early_stop]
)

from sklearn.metrics import roc_auc_score

y_prob = model.predict(X_test).ravel()
roc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", roc)

Epoch 1/110
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - auc: 0.7405 - loss: 0.6676 - val_auc: 0.8132 - val_loss: 0.4800
Epoch 2/110
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - auc: 0.7998 - loss: 0.4883 - val_auc: 0.8293 - val_loss: 0.4310
Epoch 3/110
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - auc: 0.8110 - loss: 0.4666 - val_auc: 0.8272 - val_loss: 0.4279
Epoch 4/110
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - auc: 0.8209 - loss: 0.4474 - val_auc: 0.8279 - val_loss: 0.4270
Epoch 5/110
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - auc: 0.8386 - loss: 0.4242 - val_auc: 0.8289 - val_loss: 0.4260
Epoch 6/110
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - auc: 0.8524 - loss: 0.4078 - val_auc: 0.8307 - val_loss: 0.4261
Epoch 7/110
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - auc: 0.8490 - loss: 0.4197 - val_auc: 0.8300 - val_loss: 0.4229
Epoch 8/110
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.8567 - loss: 0.4062 - val_auc: 0.8349 - val_loss: 0.4194
Epoch 9/110
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4m

In [ ]:
joblib.dump(voting_clf, "churn_model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']